# Axon Training Pipeline — Colab Notebook

**From floor plan to fabrication. Any PDF. Your products. Zero guesswork.**

This notebook runs the full Axon training pipeline on Google Colab with GPU acceleration:

1. **Setup** — Clone repo, install dependencies, mount Google Drive
2. **Data** — Upload or link training datasets
3. **MPM Pre-training** — Masked Primitive Modeling (self-supervised)
4. **SFT Fine-tuning** — Supervised fine-tuning on labeled data
5. **GRPO Annealing** — Quality annealing with group relative policy optimization
6. **DRL Training** — Deep Reinforcement Learning for panelization + placement
7. **Evaluation** — Run benchmarks on trained models

**Requirements:** Colab GPU runtime (T4 or better). Go to *Runtime > Change runtime type > GPU*.

## 1. Setup

Clone the repository, install dependencies, and mount Google Drive for persistent checkpoint storage.

In [ ]:
# Clone the Axon repository
!git clone https://github.com/tylermark/Axon.git 2>/dev/null || echo "Repo already cloned."
%cd Axon

# Install in editable mode with training extras
!pip install -e ".[dev]" -q
!pip install sb3-contrib stable-baselines3 wandb gymnasium -q

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive for persistent checkpoints and data
from google.colab import drive
drive.mount("/content/drive")

# Persistent directories on Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"
LOG_DIR = f"{DRIVE_ROOT}/logs"

import os
for d in [CKPT_DIR, DATA_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}")

print(f"\nCheckpoints will persist at: {CKPT_DIR}")
print(f"Datasets expected at:        {DATA_DIR}")

## 2. Upload Datasets

Axon trains on floor plan datasets. You have two options:

**Option A — Google Drive (recommended for large datasets):**
Upload your dataset files to `axon/datasets/` in your Google Drive before running this notebook.

**Option B — Direct upload:**
Use the cell below to upload files directly (limited to Colab session lifetime).

Expected dataset structure:
```
datasets/
  ResPlan/
    ResPlan.pkl          # ResPlan residential floor plans
  FloorPlanCAD/
    ...                  # FloorPlanCAD dataset
  MLStruct/
    ...                  # MLStruct FP dataset
```

In [ ]:
# Option A: Symlink Drive datasets into the working directory
import os
from pathlib import Path

local_data = Path("datasets")
local_data.mkdir(exist_ok=True)

drive_data = Path(DATA_DIR)
if drive_data.exists():
    for item in drive_data.iterdir():
        target = local_data / item.name
        if not target.exists():
            os.symlink(str(item), str(target))
            print(f"  Linked: {item.name}")
    print("Drive datasets linked.")
else:
    print(f"No datasets found at {DATA_DIR}. Upload data or use Option B.")

# Option B: Direct upload (uncomment to use)
# from google.colab import files
# uploaded = files.upload()  # Upload ResPlan.pkl or other dataset files

## 3. MPM Pre-training

**Masked Primitive Modeling** — self-supervised pre-training that learns structural
representations by masking 80% of vector primitives and reconstructing them.

This is Phase A of the training pipeline. Takes ~2-4 hours on a T4 GPU with default settings.

In [ ]:
from src.training.mpm import MPMPreTrainer, MPMConfig
from src.training.tracking import ExperimentTracker, CheckpointManager

# Configure MPM pre-training
mpm_config = MPMConfig(
    mask_ratio=0.8,
    epochs=100,
    batch_size=16,
    learning_rate=1e-4,
    checkpoint_dir=f"{CKPT_DIR}/mpm",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# Set up tracking
mpm_tracker = ExperimentTracker(
    project="axon-mpm",
    run_name="mpm-pretrain-colab",
    config={
        "mask_ratio": mpm_config.mask_ratio,
        "epochs": mpm_config.epochs,
        "batch_size": mpm_config.batch_size,
        "learning_rate": mpm_config.learning_rate,
    },
    enabled=True,  # Set False to disable W&B
    log_dir=f"{LOG_DIR}/mpm",
)

mpm_ckpt = CheckpointManager(
    checkpoint_dir=f"{CKPT_DIR}/mpm",
    max_checkpoints=5,
)

print(f"MPM config: {mpm_config}")
print(f"Checkpoints: {CKPT_DIR}/mpm")
print(f"Logs: {LOG_DIR}/mpm")

In [ ]:
# Run MPM pre-training
# This cell will take a while — monitor progress in the output or W&B dashboard.

trainer = MPMPreTrainer(config=mpm_config)
trainer.train(data_root="datasets/")

# Save final checkpoint to Drive
mpm_ckpt.save(
    state={
        "model_state_dict": trainer.model.state_dict(),
        "config": vars(mpm_config),
        "epoch": mpm_config.epochs,
        "metrics": {"loss": trainer.best_loss if hasattr(trainer, "best_loss") else 0.0},
    },
    epoch=mpm_config.epochs,
    metrics={"loss": trainer.best_loss if hasattr(trainer, "best_loss") else 0.0},
)

mpm_tracker.finish()
print("MPM pre-training complete. Checkpoint saved to Google Drive.")

## 4. SFT Fine-tuning

**Supervised Fine-Tuning** — fine-tune the pre-trained model on labeled floor plan
datasets (CubiCasa5K, FloorPlanCAD, etc.).

This is Phase B of the training pipeline. Requires an MPM checkpoint from Step 3.

In [ ]:
from src.training.sft import SFTTrainer, SFTConfig

# Load best MPM checkpoint
mpm_checkpoint = mpm_ckpt.load_best(metric="loss", lower_is_better=True)
if mpm_checkpoint is None:
    print("No MPM checkpoint found. Attempting to load from default path...")
    mpm_checkpoint_path = f"{CKPT_DIR}/mpm/best.pt"
else:
    print(f"Loaded MPM checkpoint (epoch {mpm_checkpoint.get('epoch', '?')})")

# Configure SFT
sft_config = SFTConfig(
    epochs=50,
    batch_size=16,
    learning_rate=5e-5,
    checkpoint_dir=f"{CKPT_DIR}/sft",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

sft_tracker = ExperimentTracker(
    project="axon-sft",
    run_name="sft-finetune-colab",
    config=vars(sft_config) if hasattr(sft_config, "__dict__") else {},
    enabled=True,
    log_dir=f"{LOG_DIR}/sft",
)

sft_ckpt = CheckpointManager(
    checkpoint_dir=f"{CKPT_DIR}/sft",
    max_checkpoints=5,
)

print(f"SFT config ready. Checkpoints: {CKPT_DIR}/sft")

In [ ]:
# Run SFT fine-tuning
sft_trainer = SFTTrainer(config=sft_config)

# Load pre-trained weights from MPM
if mpm_checkpoint and "model_state_dict" in mpm_checkpoint:
    sft_trainer.model.load_state_dict(mpm_checkpoint["model_state_dict"], strict=False)
    print("Loaded MPM pre-trained weights into SFT model.")

sft_trainer.train(data_root="datasets/")

sft_tracker.finish()
print("SFT fine-tuning complete.")

## 5. GRPO Quality Annealing

**Group Relative Policy Optimization** — uses the SFT model as a reference and
anneals output quality through a reward-based objective.

This is Phase C of the training pipeline. Requires an SFT checkpoint from Step 4.

In [ ]:
from src.training.grpo import GRPOTrainer, GRPOConfig

# Configure GRPO
grpo_config = GRPOConfig(
    iterations=1000,
    checkpoint_dir=f"{CKPT_DIR}/grpo",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

grpo_tracker = ExperimentTracker(
    project="axon-grpo",
    run_name="grpo-anneal-colab",
    config=vars(grpo_config) if hasattr(grpo_config, "__dict__") else {},
    enabled=True,
    log_dir=f"{LOG_DIR}/grpo",
)

# Load SFT checkpoint as reference
sft_checkpoint = sft_ckpt.load_best(metric="loss", lower_is_better=True)
if sft_checkpoint:
    print(f"Loaded SFT checkpoint (epoch {sft_checkpoint.get('epoch', '?')})")
else:
    print("No SFT checkpoint found. GRPO will start from scratch.")

In [ ]:
# Run GRPO annealing
grpo_trainer = GRPOTrainer(config=grpo_config)

if sft_checkpoint and "model_state_dict" in sft_checkpoint:
    grpo_trainer.model.load_state_dict(sft_checkpoint["model_state_dict"], strict=False)
    print("Loaded SFT weights as GRPO reference.")

grpo_trainer.train()

grpo_tracker.finish()
print("GRPO quality annealing complete.")

## 6. DRL Training

**Deep Reinforcement Learning** for wall panelization and pod placement.
Uses MaskablePPO to learn optimal panel selection and product placement
policies on synthetic and ResPlan floor plans.

This is Phase D of the training pipeline. Can run independently of Phases A-C.

In [ ]:
from src.training.drl_training import DRLTrainingPipeline, DRLTrainingConfig

# Configure DRL training
drl_config = DRLTrainingConfig(
    total_timesteps=500_000,
    learning_rate=3e-4,
    batch_size=64,
    n_envs=4,
    checkpoint_dir=f"{CKPT_DIR}/drl",
    device="cuda" if torch.cuda.is_available() else "auto",
    wandb_project="axon-drl",
    wandb_enabled=True,  # Set False to disable W&B
    eval_freq=10_000,
    num_eval_episodes=20,
    resplan_path=f"{DATA_DIR}/ResPlan/ResPlan.pkl",
    use_resplan=True,  # Set False if ResPlan is not available
    seed=42,
)

print("DRL Training Configuration:")
print(f"  Timesteps:    {drl_config.total_timesteps:,}")
print(f"  Envs:         {drl_config.n_envs}")
print(f"  Batch size:   {drl_config.batch_size}")
print(f"  LR:           {drl_config.learning_rate}")
print(f"  Checkpoints:  {drl_config.checkpoint_dir}")
print(f"  ResPlan:      {drl_config.use_resplan}")

In [ ]:
# Run DRL training
pipeline = DRLTrainingPipeline(config=drl_config)

try:
    pipeline.train()
    print("\nDRL training complete!")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: !pip install sb3-contrib stable-baselines3")
finally:
    pipeline.close()

## 7. Evaluation

Run benchmarks on the trained models to assess extraction quality and DRL performance.

In [ ]:
# Evaluate DRL training results
try:
    results = pipeline.evaluate()

    print("=" * 50)
    print(" DRL Evaluation Results")
    print("=" * 50)
    print(f"  Trained reward:       {results['trained_reward']:.2f} +/- {results['trained_reward_std']:.2f}")
    print(f"  Trained SPUR:         {results['trained_spur']:.3f}")
    print(f"  Trained wall cov:     {results['trained_wall_coverage_pct']:.1f}%")
    print(f"  Trained room cov:     {results['trained_room_coverage_pct']:.1f}%")
    print(f"  Trained violations:   {results['trained_violations']:.1f}")
    print()
    print(f"  Greedy reward:        {results['greedy_reward']:.2f}")
    print(f"  Greedy SPUR:          {results['greedy_spur']:.3f}")
    print(f"  Greedy wall cov:      {results['greedy_wall_coverage_pct']:.1f}%")
    print(f"  Greedy room cov:      {results['greedy_room_coverage_pct']:.1f}%")
    print()
    print(f"  Reward improvement:   {results['reward_improvement']:+.2f}")
    print(f"  SPUR improvement:     {results['spur_improvement']:+.3f}")
except RuntimeError as e:
    print(f"Evaluation not available: {e}")

In [ ]:
# List all saved checkpoints across training phases
from src.training.tracking import CheckpointManager

print("Saved Checkpoints:")
print("-" * 60)

for phase in ["mpm", "sft", "grpo", "drl"]:
    ckpt_dir = f"{CKPT_DIR}/{phase}"
    mgr = CheckpointManager(ckpt_dir)
    checkpoints = mgr.list_checkpoints()
    if checkpoints:
        print(f"\n  {phase.upper()} ({len(checkpoints)} checkpoints):")
        for ck in checkpoints:
            metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in ck.get("metrics", {}).items())
            print(f"    epoch {ck['epoch']:>4d}: {metrics_str}")
    else:
        print(f"\n  {phase.upper()}: no checkpoints")

---

**Next steps:**
- Download checkpoints from Google Drive for local inference
- Run the full Axon pipeline: `python -m src.pipeline.cli run --input plan.pdf`
- Visualize results in the Layer 1 demo notebook: `notebooks/axon_layer1_demo.ipynb`